# Demo Notebook for Molten Salt Simulator

This notebook demonstrates the use of the class `MoltenSaltSimulator` to run molecular dynamics simulations of molten salts. The class is based on the ASE package to run molecular dynamics and several uMLIPs to calculate the energy of the arrangements at the different timesteps. The results are written to ASE trajectory files for later analysis and properties can be calculated using the MoltenSaltAnalyzer class provided in the package.

## Initialization

The simulator is initialized by choosing the MLIP with the desired parameters. For reproducibility make sure to fix the random seed.

In [1]:
import os

import numpy as np

import moltensaltcalc as msc

# Set a random seed for reproducibility
np.random.seed(42)

DATA_DIR = os.path.join("demo_simulation_results")

# Load the simulator with the GRACE small MLIP with one layer. Might take a while if the model needs to be downloaded.
sim = msc.MoltenSaltSimulator(
    model_name="GRACE",  # Name of the MLIP (GRACE, FAIRCHEM, MACE)
    model_parameters={
        "model_size": "small",  # Size of the MLIP (small, medium, or large)
        "num_layers": 1,  # Number of layers in the MLIP (1 or 2)
        "model_task": "OAM",  # Which dataset the MLIP was trained on (OAM or OMAT)
    },
)

[tensorpotential] Info: Environment variable TF_USE_LEGACY_KERAS is automatically set to '1'.

Using cached GRACE model from C:\Users\isler/.cache/grace\GRACE-1L-OAM
Model license: Academic Software License


## Simulation Parameters

Define the salt composition, temperatures, initial densities, number of steps and other parameters for the simulations.

In [ ]:
# Define salts to simulate like:   "salt_name": ([anion_1, anion_2, ...], [cations], [amount_anion_1, amount_anion_2, ...], [amount_of_cations])
salts = {"NaCl": (["Cl"], ["Na"], [100], [100])}
lattice_type = "rocksalt"  # Initial structure to be generated. Can be "random" for random placement or "rocksalt" for an initial structure with a rocksalt lattice
random_removal = False  # If true and lattice is "rocksalt", randomly remove excess atoms to match the desired composition. If false, simply take the first N_atoms positions from the generated lattice to match the desired composition.

n_steps = 10  # How many timesteps to run the simulation for. In practice ~100'000 steps
print_status = True  # Whether to print the status of the simulation
print_interval = 2  # How often to print the status. In practice ~10 steps
write_interval = 2  # How often to write the trajectory frames. In practice ~10 steps
timestep_fs = 10.0  # How long one timestep is in fs. In practice ~1 fs
taut_fs = 100.0  # Time constant for Berendsen temperature coupling in fs. In practice ~100 fs
taup_fs = 1000.0  # Time constant for Berendsen pressure coupling in fs. In practice ~1000 fs
tdamp_fs = 1000.0  # Characteristic time scale for NoseHooverChainNVT thermostat in fs, typically 100*timestep_fs. In practice ~100 fs
compressibility_per_bar = 4.0e-5  # Compressibility of the system per bar in 1/bar. In practice ~4.0e-5
pressure_bar = 1.01325  # Pressure in bar. In practice ~1.01325 bar
logfile_npt = os.path.join(DATA_DIR, "npt_run.log")  # Logfile for the NPTBerendsen dynamics simulation, "-" for stdout
logfile_nvt = os.path.join(
    DATA_DIR, "nvt_run.log"
)  # Logfile for the NoseHooverChainNVT dynamics simulation, "-" for stdout

# Define at which temperatures you want to calculate the properties per salt
temperatures = {"NaCl": [1100, 1150, 1200]}
# Define what density you guess the salt to have at the corresponding temperatures
density_guesses = {"NaCl": [1.542, 1.515, 1.488]}

## Run the Simulations

The usual workflow is to first run an NPT (constant particles, pressure and temperature) simulation to equilibrate the system volume and temperature, and the run a NVT (constant particles, volume and temperature) simulation, from which many properties can be calculated. The following code shows how to run the simulations for the NaCl salt with the GRACE MLIP and the parameters defined above.

In [ ]:
for salt_name, (anions, cations, n_anions, n_cations) in salts.items():
    print(f"\nRunning NPT simulations for {salt_name}...\n")

    # Create folders to store the trajectories
    npt_dir, nvt_dir = sim.create_simulation_folder(
        base_name=os.path.join(DATA_DIR, f"GRACE_1L_{salt_name}_super_short")
    )

    # Pair each temperature with its corresponding density guess
    for T, density_guess in zip(temperatures[salt_name], density_guesses[salt_name], strict=False):
        # Create the initial system
        atoms = sim.build_system(
            anions,
            cations,
            n_anions,
            n_cations,
            density_guess,
            lattice=lattice_type,
            random_removal=random_removal,
        )
        # Run the NPT simulation
        traj_file_npt = os.path.join(npt_dir, f"npt_{salt_name}_{T}K.traj")
        atoms = sim.run_npt_simulation(
            atoms,
            T,
            steps=n_steps,
            print_status=print_status,
            print_interval=print_interval,
            write_interval=write_interval,
            timestep_fs=timestep_fs,
            taut_fs=taut_fs,
            taup_fs=taup_fs,
            compressibility_per_bar=compressibility_per_bar,
            pressure_bar=pressure_bar,
            traj_file=traj_file_npt,
            logfile=logfile_npt,
        )
        # Run the NVT simulation
        traj_file_nvt = os.path.join(nvt_dir, f"nvt_{salt_name}_{T}K.traj")
        sim.run_nvt_simulation(
            atoms,
            T,
            steps=n_steps,
            print_status=print_status,
            print_interval=print_interval,
            write_interval=write_interval,
            timestep_fs=timestep_fs,
            tdamp_fs=tdamp_fs,
            traj_file=traj_file_nvt,
            logfile=logfile_nvt,
        )


Running NPT simulations for NaCl...

Simulation folders created in: c:\Users\isler\OneDrive - DIFFER, Dutch Institute for Fundamental Energy Research\Desktop\PhD Projects\MS_Prediction_Project1\MoltenSaltCalc\demo\demo_simulation_results\GRACE_1L_NaCl_super_short
Step      0 | T = 1100.000000 K | P = -9.174895e-03 bar | V =  6293.22 Å³
Step      2 | T = 1014.489359 K | P = -5.664542e-03 bar | V =  6247.79 Å³
Step      4 | T = 810.726324 K | P = 1.669848e-03 bar | V =  6238.62 Å³
Step      6 | T = 874.835178 K | P = 4.938636e-03 bar | V =  6280.03 Å³
Step      8 | T = 1051.522667 K | P = 4.805523e-03 bar | V =  6340.72 Å³
Step     10 | T = 1086.015794 K | P = 5.399506e-03 bar | V =  6404.32 Å³
NPT simulation finished, trajectory saved to c:\Users\isler\OneDrive - DIFFER, Dutch Institute for Fundamental Energy Research\Desktop\PhD Projects\MS_Prediction_Project1\MoltenSaltCalc\demo\demo_simulation_results\GRACE_1L_NaCl_super_short\NPT\npt_NaCl_1100K.traj
Step      0 | T = 1100.000000 K 

The trajectory files are now saved in the folder `demo_simulation_results` for futher analysis (see `demo/analyzer.ipynb`). The logs for the simulation are saved in the folder `demo_simulation_results/` as well.